# Blue Kangaroo Research Project

## Setup

### Environment Setup

For this project, the following packages are currently in use:

1. numpy
    1. Used for basic math.
2. pandas
    1. Used for planning and in-process data handling.
3. matplotlib
    1. Used to display data.
4. ipympl
    1. Used to make plots interactive.
5. tensorflow
    1. Used for basic machine learning.
6. keras
    1. Used for neural networks.
7. duckdb
    1. Used for data handling.
8. sqlalchemy
    1. Used to make a duckdn ORM.
9. duckdb-engine
    1. Used to connect duckdb to sqlalchemy.

In [1]:
%pip install numpy pandas matplotlib ipympl tensorflow keras duckdb sqlalchemy duckdb-engine
%matplotlib widget

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Program

### Basic Imports

In [2]:
import numpy as np
import pandas as pd
from pandas import DataFrame, read_parquet, read_csv
from matplotlib import pyplot as plt
from pathlib import Path
from sqlalchemy import create_engine, URL, Engine, text, VARCHAR, BOOLEAN, BIGINT, SMALLINT, TIMESTAMP, INTEGER, FLOAT, Table, Column, ForeignKey, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker, relationship
from typing import List

### Data Intake

#### Table Paths

In [3]:
base_path: Path = Path("/data/shared/OpenAlex/processed-snapshots/")
parquet_path: Path = base_path / "parquet-files" / "may-2025"
csv_path: Path = base_path / "csv-files" / "may-2025"
works_path: Path = parquet_path / "works"
topics_path: Path = parquet_path / "works_topics"
referenced_works_path: Path = parquet_path / "works_referenced_works"
keywords_path: Path = parquet_path / "works_keywords"
concepts_path: Path = parquet_path / "works_concepts"

#### Tables

In [4]:
class Base(DeclarativeBase):
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({", ".join(filter(lambda value: value is not None, map(lambda key: f"{key}={self.__dict__[key]}" if self.__dict__[key] is not None else None, self.__mapper__.columns.keys())))})" # type: ignore
    def to_dict(self) -> dict:
        return dict(map(lambda key: (key, self.__dict__[key]), self.__mapper__.columns.keys()))

referenced_works: Table = Table(
    "referenced_works",
    Base.metadata,
    Column("work_id", BIGINT, ForeignKey("Work.work_id"), primary_key = True),
    Column("referenced_work_id", BIGINT, ForeignKey("Work.work_id"), primary_key = True),
)

class Work(Base):
    __tablename__ = "works"
    work_id: Mapped[int] = mapped_column(BIGINT, primary_key = True)
    doi: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    title: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    publication_year: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    publication_date: Mapped[str] = mapped_column(TIMESTAMP, nullable = True)
    type: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    type_crossref: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    cited_by_count: Mapped[int] = mapped_column(INTEGER, nullable = True)
    num_authors: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    num_locations: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    num_references: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    language: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    has_complete_institution_info: Mapped[bool] = mapped_column(BOOLEAN, nullable = True)
    has_grant_info: Mapped[bool] = mapped_column(BOOLEAN, nullable = True)
    has_keywords: Mapped[bool] = mapped_column(BOOLEAN, nullable = True)
    is_retracted: Mapped[bool] = mapped_column(BOOLEAN, nullable = True)
    is_paratext: Mapped[bool] = mapped_column(BOOLEAN, nullable = True)
    created_date: Mapped[str] = mapped_column(TIMESTAMP, nullable = True)
    gz_path: Mapped[str] = mapped_column(VARCHAR, nullable = True)

class Topic(Base):
    __tablename__ = "topics"
    work_id: Mapped[int] = mapped_column(ForeignKey(Work.work_id), nullable = True)
    publication_year: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    is_primary_topic: Mapped[bool] = mapped_column(BOOLEAN, nullable = True)
    score: Mapped[float] = mapped_column(FLOAT, nullable = True)
    topic_id: Mapped[int] = mapped_column(INTEGER, primary_key = True)
    topic_name: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    subfield_id: Mapped[int] = mapped_column(INTEGER, nullable = True)
    subfield_name: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    field_id: Mapped[int] = mapped_column(INTEGER, nullable = True)
    field_name: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    domain_id: Mapped[int] = mapped_column(INTEGER, nullable = True)
    domain_name: Mapped[str] = mapped_column(VARCHAR, nullable = True)

class Keyword(Base):
    __tablename__ = "keywords"
    work_id: Mapped[int] = mapped_column(ForeignKey("Work.work_id"), nullable = True)
    publication_year: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    keyword: Mapped[str] = mapped_column(VARCHAR, primary_key = True)
    score: Mapped[float] = mapped_column(FLOAT, nullable = True)

class Concept(Base):
    __tablename__ = "concepts"
    work_id: Mapped[int] = mapped_column(ForeignKey("Work.work_id"), nullable = True)
    publication_year: Mapped[int] = mapped_column(SMALLINT, nullable = True)
    score: Mapped[float] = mapped_column(FLOAT, nullable = True)
    concept_id: Mapped[int] = mapped_column(BIGINT, primary_key = True)
    concept_name: Mapped[str] = mapped_column(VARCHAR, nullable = True)
    level: Mapped[int] = mapped_column(SMALLINT, nullable = True)

#### Database Connection

In [5]:
url: URL = URL.create(drivername = "duckdb", database = ":memory:")
eng: Engine = create_engine(url)
Session: sessionmaker = sessionmaker(eng)
def start_session():
    session = Session()
    session.execute(text(f"""
        SET memory_limit = '4GB';
        SET enable_progress_bar = false;
        SET enable_progress_bar_print = false;
        SET preserve_insertion_order = false;
        CREATE VIEW works AS FROM read_parquet('{works_path / "*.parquet"}');
        CREATE VIEW topics AS FROM read_parquet('{topics_path / "*.parquet"}');
        CREATE VIEW referenced_works AS FROM read_parquet('{referenced_works_path / "*.parquet"}');
        CREATE VIEW keywords AS FROM read_parquet('{keywords_path / "*.parquet"}');
        CREATE VIEW concepts AS FROM read_parquet('{concepts_path / "*.parquet"}');
    """))
    return session

#### Collection

In [6]:
with start_session() as session:
    data = pd.DataFrame(session.query(Topic.work_id, Topic.topic_id).filter(Topic.publication_year == 2000).filter(Topic.work_id.in_(session.query(Topic.work_id).filter(Topic.subfield_name == "Artificial Intelligence").distinct().scalar_subquery())).distinct().all())

In [7]:
data

,work_id,topic_id
0,2370850869,13442
1,2370850869,12157
2,2370850869,11740
3,2371051693,10715
4,2371051693,13734
...,...,...
198570,619300482,10688
198571,619300482,10299
198572,929770819,11596
198573,929770819,10215


In [ ]:
transactions = pd.DataFrame([[work, *([False] * data["topic_id"].unique().size)] for work in data["work_id"].unique()], columns = ["work", *data["topic_id"].unique()])
with start_session() as session:
    for topic in data["topic_id"].unique():
        transactions[topic] = transactions["work"].apply(lambda work: len(session.query(Topic).filter(Topic.work_id == int(work)).filter(Topic.topic_id == int(topic)).limit(1).all()) > 0)

#### Process

I was not able to get a full model running in time. The above cells are intended to turn all of the works written in 2000, which have Artificial Intelligence as a topic, into one-hot encoded transactions. The number of topics in question is just over 2000, so the idea was to one-hot encode each work by the topics associated with it, in order to both identify association rules and to incorporate more information into a machine learning algorithm. I would then move through various years to create a model of the association rules over time and clusters. Once this basic process is complete, the idea is to then find association rules between keywords and topics, topics and concepts, etc...

I severely underestimated the sheer amount of time it would take to process the data. This one-hot encoding from 2000 alone has been running for over 30 minutes, and has so far provided nothing usable. Of the various years I looked through, 2000 was on the smaller end of works, which is why I wanted to use it.

Due to the issues I had implementing this, I have decided to take up a larger portion of the remaining portion of the project.

### Data Visualization

In [ ]:
# plt.subplot(1, 1, 1)
plt.tight_layout()
plt.show()